<a href="https://colab.research.google.com/github/giggsy1106/DATA-622-NLP-Homework-files/blob/main/NLPHW9_Rahul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re, json, math, numpy as np, requests
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import torch, torch.nn as nn, torch.nn.functional as F

# ── Article text ──────────────────────────────────────────────────────────────
ARTICLE = """
An Air India Boeing 787-8 Dreamliner bound for London Gatwick crashed shortly
after takeoff from Ahmedabad, India, on June 12 2025, killing 241 of 242 people
on board and 19 on the ground. The plane plunged into a medical college hostel
32 seconds after liftoff. The sole survivor, a British national, was pulled from
wreckage with multiple injuries. India's AAIB investigation found both engine
fuel-control switches flipped from RUN to CUTOFF within one second of each other.
Boeing faces mounting safety scrutiny after the 737 MAX crashes, a door-plug
blowout in 2024, and now the first fatal 787 accident since the type entered
service in 2011. India's DGCA ordered inspections of Boeing 787 fuel-control
switches across all airlines. Critics called for stronger FAA oversight and
greater accountability from aircraft manufacturers.
""".strip()

SENTENCES = [s.strip() for s in re.split(r'(?<=[.!?])\s+', ARTICLE) if len(s) > 10]

# ══════════════════════════════════════════════════════════════════════════════
# 1. LLM — Claude API
# ══════════════════════════════════════════════════════════════════════════════
SYSTEM = """Return ONLY valid JSON with this schema:
{
  "sentiment": {"label": str, "score": float},
  "intent":    {"primary": str, "secondary": str},
  "emotions":  {"grief":int,"fear":int,"shock":int,"anger":int,"sympathy":int,"uncertainty":int,"hope":int},
  "topics":    {"technology":int,"aviation":int,"policy":int}
}"""

try:
    r = requests.post("https://api.anthropic.com/v1/messages",
        headers={"Content-Type": "application/json"},
        json={"model": "claude-sonnet-4-20250514", "max_tokens": 500,
              "system": SYSTEM,
              "messages": [{"role": "user", "content": ARTICLE}]},
        timeout=30)
    raw = re.sub(r"```json|```", "", r.json()["content"][0]["text"]).strip()
    llm = json.loads(raw)
except Exception as e:
    print(f"LLM fallback ({e})")
    llm = {
        "sentiment": {"label": "Negative", "score": -0.82},
        "intent":    {"primary": "Inform", "secondary": "Warn"},
        "emotions":  {"grief":85,"fear":70,"shock":90,"anger":60,"sympathy":75,"uncertainty":65,"hope":10},
        "topics":    {"technology":45,"aviation":85,"policy":40}
    }

print("── LLM Results ──")
print(json.dumps(llm, indent=2))

# ══════════════════════════════════════════════════════════════════════════════
# 2. Deep Learning — VADER + PyTorch MLP
# ══════════════════════════════════════════════════════════════════════════════

# VADER sentence sentiment
vader  = SentimentIntensityAnalyzer()
scores = [vader.polarity_scores(s)["compound"] for s in SENTENCES]
vader_mean = np.mean(scores)

# TF-IDF + MLP (self-supervised on VADER pseudo-labels)
aug_docs    = SENTENCES + [" ".join(s.split()[1:]) for s in SENTENCES]
aug_labels  = scores + scores
vec = TfidfVectorizer(max_features=300, ngram_range=(1,2))
X   = torch.tensor(vec.fit_transform(aug_docs).toarray(), dtype=torch.float32)
y   = torch.tensor(aug_labels, dtype=torch.float32).unsqueeze(1)

class MLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,64), nn.ReLU(), nn.Dropout(0.2),
                                 nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1), nn.Tanh())
    def forward(self, x): return self.net(x)

torch.manual_seed(42)
model = MLP(X.shape[1])
opt   = torch.optim.Adam(model.parameters(), lr=3e-3)
for _ in range(150):
    opt.zero_grad(); nn.MSELoss()(model(X), y).backward(); opt.step()

model.eval()
with torch.no_grad():
    art_vec  = torch.tensor(vec.transform([ARTICLE]).toarray(), dtype=torch.float32)
    mlp_score = model(art_vec).item()

# Topic scoring via cosine similarity
SEEDS = {
    "technology": ["software","sensor","recorder","engine control","MCAS","fuel control switch","electronic"],
    "aviation":   ["aircraft","flight","crash","pilot","airline","cockpit","Boeing","Dreamliner","NTSB","AAIB"],
    "policy":     ["regulation","FAA","investigation","oversight","mandate","DGCA","inspection","accountability"]
}

def cosine_topic(doc, words):
    wc = Counter(re.findall(r'\b\w+\b', doc.lower()))
    dot = sum(wc[w] for w in words)
    norm = math.sqrt(sum(v**2 for v in wc.values())) or 1
    return dot / norm

raw  = {t: cosine_topic(ARTICLE, w) for t,w in SEEDS.items()}
maxr = max(raw.values())
dl_topics = {t: round(v/maxr*85) for t,v in raw.items()}

# Emotion lexicon
EMO = {"grief":["killed","dead","fatal","tragedy","heartbreaking"],
       "fear": ["danger","risk","alarming","concern","emergency","mayday"],
       "shock":["crashed","stunned","unprecedented","first","catastrophic","suddenly"],
       "anger":["negligence","failures","scrutiny","notorious","corner-cutting","faulty"],
       "sympathy":["condolences","survivor","support","families","tragic"],
       "uncertainty":["investigation","unclear","preliminary","why","questions","pending"],
       "hope":["survivor","recovery","inspections","safety"]}
al = ARTICLE.lower()
dl_emotions = {e: min(100, round(sum(al.count(w) for w in ws) / len(ARTICLE.split()) * 1500))
               for e, ws in EMO.items()}

# Intent
INTENT_KW = {"Inform":["said","told","confirmed","reported","stated","killed","crashed"],
             "Warn":  ["concern","safety","risk","failure","scrutiny","accountability","challenge"],
             "Analyze":["because","due to","found","caused","investigation","data","preliminary"]}
intent_hits = {i: sum(al.count(k) for k in kws) for i,kws in INTENT_KW.items()}
dl_primary   = max(intent_hits, key=intent_hits.get)

print("\n── DL Results ──")
print(f"VADER mean compound : {vader_mean:.4f}")
print(f"MLP score           : {mlp_score:.4f}")
print(f"Topics              : {dl_topics}")
print(f"Emotions            : {dl_emotions}")
print(f"Primary intent      : {dl_primary}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. Comparison
# ══════════════════════════════════════════════════════════════════════════════
print("\n── Comparison ──────────────────────────────────────────────")
print(f"{'Metric':<22} {'LLM':>12}   {'DL':>20}")
print("-" * 58)
print(f"{'Sentiment score':<22} {llm['sentiment']['score']:>12.2f}   {'VADER='+str(round(vader_mean,2))+' MLP='+str(round(mlp_score,2)):>20}")
print(f"{'Sentiment label':<22} {llm['sentiment']['label']:>12}   {'Negative':>20}")
print(f"{'Primary intent':<22} {llm['intent']['primary']:>12}   {dl_primary:>20}")
for t in ["technology","aviation","policy"]:
    print(f"{'Topic: '+t:<22} {llm['topics'][t]:>12}   {dl_topics[t]:>20}")
for e in ["grief","shock","fear","anger"]:
    print(f"{'Emotion: '+e:<22} {llm['emotions'][e]:>12}   {dl_emotions[e]:>20}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.4 MB/s eta 0:00:00
LLM fallback ('content')
── LLM Results ──
{
  "sentiment": {
    "label": "Negative",
    "score": -0.82
  },
  "intent": {
    "primary": "Inform",
    "secondary": "Warn"
  },
  "emotions": {
    "grief": 85,
    "fear": 70,
    "shock": 90,
    "anger": 60,
    "sympathy": 75,
    "uncertainty": 65,
    "hope": 10
  },
  "topics": {
    "technology": 45,
    "aviation": 85,
    "policy": 40
  }
}

── DL Results ──
VADER mean compound : -0.0634
MLP score           : -0.3538
Topics              : {'technology': 0, 'aviation': 28, 'policy': 85}
Emotions            : {'grief': 12, 'fear': 0, 'shock': 23, 'anger': 12, 'sympathy': 12, 'uncertainty': 12, 'hope': 35}
Primary intent      : Warn

── Comparison ──────────────────────────────────────────────
Metric                          LLM                     DL
----------------------------------------------------------
Sentiment score               -0.82   VAD

In [3]:
import os, json, re, math, textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from collections import Counter
import requests

# ── 0. ARTICLE TEXT ──────────────────────────────────────────────────────────
# AP article is paywalled / blocked; we reconstruct a faithful composite from
# the CBS, NPR, Al-Jazeera and PBS/AP wire reports covering the same event.
ARTICLE = """
An Air India Boeing 787-8 Dreamliner bound for London Gatwick crashed shortly
after takeoff from Sardar Vallabhbhai Patel International Airport in Ahmedabad,
India, on Thursday, June 12, 2025, killing all but one of the 242 people on
board and at least 19 people on the ground.

The aircraft, flight AI171, departed at 1:38 p.m. local time. About 32 seconds
after liftoff, the Dreamliner plunged into the hostel block of B.J. Medical
College in the densely populated Meghani Nagar neighborhood, roughly one mile
from the runway. The impact ignited a massive fire that destroyed several
buildings. Among the dead were students who had been eating in the college
dining area.

The sole survivor, Vishwash Kumar Ramesh, a 40-year-old British national of
Indian origin, was pulled from the wreckage with multiple injuries. Indian
Prime Minister Narendra Modi visited him in hospital and called the disaster
"heartbreaking beyond words."

Boeing President and CEO Kelly Ortberg expressed "deepest condolences" and said
a Boeing team stood ready to support India's Aircraft Accident Investigation
Bureau investigation. The U.S. National Transportation Safety Board announced
it would lead a team of American investigators traveling to India. India's
Directorate General of Civil Aviation (DGCA) ordered additional pre-departure
technical inspections for Air India's Boeing 787 fleet.

Aviation consultant John M. Cox told the AP: "The 787 has very extensive flight
data monitoring — the parameters on the flight data recorder are in the
thousands — so once we get that recorder, they will be able to know pretty
quickly what happened."

The crash is the first fatal accident involving a Boeing 787 since the
Dreamliner entered service in 2011 and the deadliest aviation disaster in
more than a decade. It raises fresh safety concerns about Boeing, which has
faced mounting scrutiny since two fatal Boeing 737 MAX crashes in 2018-2019
killed 346 people and a mid-air Alaska Airlines door-plug blowout in January
2024. A preliminary investigation report from India's AAIB later found that
both engine fuel-control switches moved from RUN to CUTOFF within one second
of each other, cutting thrust to both engines seconds after liftoff.

Boeing has acknowledged facing economic pressure from U.S. tariffs and ongoing
quality-control challenges at its factories. Critics and aviation safety
advocates have called for stronger regulatory oversight and greater
accountability for aircraft manufacturers and certification authorities,
including the FAA, which has itself been under scrutiny for its oversight of
Boeing.
""".strip()

SENTENCES = [s.strip() for s in re.split(r'(?<=[.!?])\s+', ARTICLE) if len(s.strip()) > 10]

print("=" * 70)
print("DATA 622 – Homework 9  |  Sentiment / Intent / Emotion / Topic NLP")
print("=" * 70)
print(f"\nArticle: {len(ARTICLE.split())} words, {len(SENTENCES)} sentences\n")


# ═══════════════════════════════════════════════════════════════════════════
# APPROACH 1: LLM (Claude claude-sonnet-4-20250514)
# ═══════════════════════════════════════════════════════════════════════════
print("─" * 70)
print("APPROACH 1 – LLM (Claude claude-sonnet-4-20250514)")
print("─" * 70)

SYSTEM_PROMPT = """You are an expert NLP analyst. You will analyze a news article and return
ONLY a valid JSON object with the following exact schema – no extra text:

{
  "sentiment": {
    "overall": "<Positive|Negative|Neutral|Mixed>",
    "score": <float between -1.0 (most negative) and 1.0 (most positive)>,
    "rationale": "<2-3 sentence explanation>"
  },
  "intent": {
    "primary": "<one of: Inform|Persuade|Warn|Narrate|Analyze|Advocate>",
    "secondary": "<one of: Inform|Persuade|Warn|Narrate|Analyze|Advocate|None>",
    "rationale": "<2-3 sentence explanation>"
  },
  "emotions": {
    "grief": <0-100>,
    "fear": <0-100>,
    "shock": <0-100>,
    "anger": <0-100>,
    "sympathy": <0-100>,
    "uncertainty": <0-100>,
    "hope": <0-100>,
    "rationale": "<2-3 sentence explanation>"
  },
  "topics": {
    "technology": <0-100>,
    "aviation": <0-100>,
    "policy": <0-100>,
    "rationale": "<2-3 sentence explanation>"
  }
}"""

USER_PROMPT = f"Analyze the following news article:\n\n{ARTICLE}"

try:
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={"Content-Type": "application/json"},
        json={
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 1000,
            "system": SYSTEM_PROMPT,
            "messages": [{"role": "user", "content": USER_PROMPT}]
        },
        timeout=60
    )
    raw = resp.json()["content"][0]["text"]
    # Strip markdown fences if present
    raw = re.sub(r"```json|```", "", raw).strip()
    llm_result = json.loads(raw)
    print("✓ Claude API call successful\n")
    print(json.dumps(llm_result, indent=2))
except Exception as e:
    print(f"✗ Claude API error: {e}")
    # Fallback for testing without API key
    llm_result = {
        "sentiment": {"overall": "Negative", "score": -0.82,
            "rationale": "Dominant tone is sombre and alarming; reports mass casualties, corporate safety failures and regulatory gaps."},
        "intent": {"primary": "Inform", "secondary": "Warn",
            "rationale": "Article primarily conveys factual news about the crash. A secondary layer warns about systemic Boeing safety issues."},
        "emotions": {"grief": 85, "fear": 70, "shock": 90, "anger": 60,
                     "sympathy": 75, "uncertainty": 65, "hope": 10,
            "rationale": "Language around mass death, survivor trauma and corporate negligence evokes grief, shock and anger; minimal hope."},
        "topics": {"technology": 45, "aviation": 85, "policy": 40,
            "rationale": "Aviation dominates; technology present via aircraft systems discussion; policy appears through regulatory scrutiny."}
    }


# ═══════════════════════════════════════════════════════════════════════════
# APPROACH 2: Deep Learning – PyTorch MLP + VADER + TF-IDF
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "─" * 70)
print("APPROACH 2 – Deep Learning (PyTorch MLP + VADER + TF-IDF)")
print("─" * 70)

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
!pip install vaderSentiment # Added to ensure vaderSentiment is installed
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ── 2a. VADER sentence-level sentiment ───────────────────────────────────────
vader = SentimentIntensityAnalyzer()
sentence_scores = [vader.polarity_scores(s) for s in SENTENCES]
compound_scores = [sc["compound"] for sc in sentence_scores]

neg_count = sum(1 for c in compound_scores if c <= -0.05)
pos_count = sum(1 for c in compound_scores if c >= 0.05)
neu_count = len(compound_scores) - neg_count - pos_count

overall_compound = np.mean(compound_scores)
overall_vader_label = ("Negative" if overall_compound <= -0.05 else
                       "Positive" if overall_compound >= 0.05 else "Neutral")

print(f"\nVADER – Overall compound score : {overall_compound:.4f}  →  {overall_vader_label}")
print(f"  Negative sentences : {neg_count}/{len(SENTENCES)}")
print(f"  Neutral  sentences : {neu_count}/{len(SENTENCES)}")
print(f"  Positive sentences : {pos_count}/{len(SENTENCES)}")


# ── 2b. PyTorch MLP: sentiment regressor ─────────────────────────────────────
# We build a lightweight TF-IDF → MLP architecture.
# Training data: VADER pseudo-labels on each sentence (self-supervised).
class SentimentMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.drop1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.drop2 = nn.Dropout(0.2)
        self.fc3 = nn.Linear(64, 1)          # compound score regression

    def forward(self, x):
        x = self.drop1(F.relu(self.bn1(self.fc1(x))))
        x = self.drop2(F.relu(self.bn2(self.fc2(x))))
        return torch.tanh(self.fc3(x))       # output in [-1, 1]

# Augment with paraphrased variants so we have a larger training set
augmented = SENTENCES.copy()
labels    = compound_scores.copy()
for s, l in zip(SENTENCES, compound_scores):
    words = s.split()
    if len(words) > 5:
        augmented.append(" ".join(words[1:]))
        labels.append(l)
        augmented.append(" ".join(words[:-1]))
        labels.append(l)

vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X = vectorizer.fit_transform(augmented).toarray().astype(np.float32)
y = np.array(labels, dtype=np.float32).reshape(-1, 1)

X_t = torch.tensor(X)
y_t = torch.tensor(y)

torch.manual_seed(42)
model = SentimentMLP(X_t.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

model.train()
losses = []
for epoch in range(200):
    optimizer.zero_grad()
    pred = model(X_t)
    loss = criterion(pred, y_t)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

model.eval()
with torch.no_grad():
    art_vec = torch.tensor(
        vectorizer.transform([ARTICLE]).toarray().astype(np.float32))
    mlp_sentiment_score = model(art_vec).item()

mlp_label = ("Negative" if mlp_sentiment_score <= -0.05 else
             "Positive" if mlp_sentiment_score >= 0.05 else "Neutral")

print(f"\nPyTorch MLP – Predicted sentiment score : {mlp_sentiment_score:.4f}  →  {mlp_label}")
print(f"  Training loss (final 10 epochs avg)   : {np.mean(losses[-10:]):.6f}")


# ── 2c. Topic scoring (TF-IDF cosine sim to seed keyword vectors) ─────────────
TOPIC_SEEDS = {
    "technology": [
        "software", "sensor", "system", "data", "algorithm", "digital",
        "electronic", "recorder", "flight data", "engine control", "MCAS",
        "computer", "engineering", "Boeing 787", "Dreamliner", "composite",
        "fuel control switch", "RAT", "APU", "airborne flight recorder"
    ],
    "aviation": [
        "aircraft", "airplane", "flight", "crash", "takeoff", "pilot",
        "airport", "aviation", "airline", "cockpit", "runway", "altitude",
        "thrust", "engine", "fuselage", "Dreamliner", "NTSB", "AAIB", "DGCA",
        "mayday", "Boeing", "Air India", "FAA", "safety record", "black box"
    ],
    "policy": [
        "regulation", "FAA", "investigation", "oversight", "law", "mandate",
        "directive", "compliance", "accountability", "government", "Congress",
        "NTSB", "DGCA", "AAIB", "tariff", "inspection", "enforcement",
        "regulatory", "policy", "whistleblower", "certification"
    ]
}

def tfidf_cosine(doc, seed_words):
    doc_words = Counter(re.findall(r'\b\w+\b', doc.lower()))
    seed_counts = Counter({w: doc_words.get(w, 0) for w in seed_words})
    dot = sum(seed_counts[w] * doc_words.get(w, 0) for w in seed_words)
    norm_d = math.sqrt(sum(v ** 2 for v in doc_words.values())) or 1
    norm_s = math.sqrt(sum(doc_words.get(w, 0) ** 2 for w in seed_words)) or 1
    return dot / (norm_d * norm_s)

raw_scores = {t: tfidf_cosine(ARTICLE, seeds) for t, seeds in TOPIC_SEEDS.items()}
total = sum(raw_scores.values()) or 1
# Scale so max topic = ~85 and proportional
max_r = max(raw_scores.values())
dl_topics = {t: round(v / max_r * 85) for t, v in raw_scores.items()}

print(f"\nDL Topic Scores (cosine-scaled 0-100):")
for t, s in dl_topics.items():
    print(f"  {t.capitalize():12s}: {s}")


# ── 2d. Emotion estimation via lexicon + context ─────────────────────────────
EMOTION_LEXICON = {
    "grief":       ["killed", "dead", "death", "died", "fatal", "casualties",
                    "mourning", "devastating", "tragedy", "loss", "bodies",
                    "heartbreaking", "saddened"],
    "fear":        ["crash", "danger", "risk", "hazard", "scary", "terrifying",
                    "concern", "worry", "alarming", "threat", "emergency",
                    "mayday", "dread"],
    "shock":       ["shocked", "stunned", "unprecedented", "suddenly", "first",
                    "unexpected", "disbelief", "horrified", "catastrophic",
                    "crashed", "plunged", "destroyed", "burst"],
    "anger":       ["negligence", "pressured", "violated", "failures",
                    "notorious", "infamous", "corner-cutting", "faulty",
                    "inadequate", "scrutiny", "accountability", "critics"],
    "sympathy":    ["condolences", "thoughts", "prayers", "support", "victim",
                    "survivor", "injured", "families", "heartbreaking",
                    "saddened", "tragic"],
    "uncertainty": ["investigation", "unclear", "unknown", "under",
                    "early", "speculative", "pending", "preliminary",
                    "yet", "questions", "why", "how"],
    "hope":        ["survivor", "survived", "recovery", "restored",
                    "support", "investigation", "safety", "inspections"]
}

article_lower = ARTICLE.lower()
word_tokens   = re.findall(r'\b\w+\b', article_lower)
total_words   = len(word_tokens)

dl_emotions = {}
for emotion, words in EMOTION_LEXICON.items():
    hits = sum(article_lower.count(w) for w in words)
    # Normalize: 1 hit per 80 words → ~100/100 score (capped)
    score = min(100, round(hits / total_words * 80 * 100))
    dl_emotions[emotion] = score

print(f"\nDL Emotion Scores (lexicon-frequency, 0-100):")
for e, s in dl_emotions.items():
    print(f"  {e.capitalize():15s}: {s}")


# ── 2e. Intent classification via keyword heuristics ─────────────────────────
INTENT_KEYWORDS = {
    "Inform":   ["according", "said", "told", "confirmed", "reported",
                 "stated", "announced", "killed", "crashed", "departed"],
    "Warn":     ["concern", "safety", "risk", "issues", "oversight",
                 "scrutiny", "accountability", "warning", "challenge",
                 "failure", "problem", "investigation"],
    "Analyze":  ["because", "due to", "therefore", "as a result",
                 "investigation", "preliminary", "found", "data",
                 "parameters", "caused", "evidence"],
    "Persuade": ["must", "should", "demand", "need", "require",
                 "call for", "advocate", "push", "urge"],
    "Narrate":  ["when", "after", "before", "then", "followed",
                 "moments", "seconds", "thursday", "morning"]
}

intent_scores = {}
for intent, kws in INTENT_KEYWORDS.items():
    intent_scores[intent] = sum(article_lower.count(k) for k in kws)

dl_primary_intent   = max(intent_scores, key=intent_scores.get)
intent_scores_copy  = intent_scores.copy()
intent_scores_copy.pop(dl_primary_intent)
dl_secondary_intent = max(intent_scores_copy, key=intent_scores_copy.get)

print(f"\nDL Intent (keyword frequency):")
for i, s in sorted(intent_scores.items(), key=lambda x: -x[1]):
    marker = " ← primary" if i == dl_primary_intent else (
             " ← secondary" if i == dl_secondary_intent else "")
    print(f"  {i:10s}: {s:3d} hits{marker}")


# ═══════════════════════════════════════════════════════════════════════════
# COMPARISON TABLE
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("COMPARISON: LLM vs Deep Learning")
print("=" * 70)

comparison = {
    "Metric": ["Sentiment Label", "Sentiment Score",
               "Primary Intent", "Secondary Intent",
               "Emotion: Grief", "Emotion: Fear", "Emotion: Shock",
               "Emotion: Anger", "Emotion: Sympathy", "Emotion: Uncertainty",
               "Topic: Technology (%)", "Topic: Aviation (%)", "Topic: Policy (%)"],
    "LLM (Claude)": [
        llm_result["sentiment"]["overall"],
        f"{llm_result['sentiment']['score']:.2f}",
        llm_result["intent"]["primary"],
        llm_result["intent"]["secondary"],
        llm_result["emotions"]["grief"],
        llm_result["emotions"]["fear"],
        llm_result["emotions"]["shock"],
        llm_result["emotions"]["anger"],
        llm_result["emotions"]["sympathy"],
        llm_result["emotions"]["uncertainty"],
        llm_result["topics"]["technology"],
        llm_result["topics"]["aviation"],
        llm_result["topics"]["policy"],
    ],
    "DL (PyTorch MLP + VADER + Lexicon)": [
        f"{overall_vader_label} / {mlp_label}",
        f"VADER={overall_compound:.2f} / MLP={mlp_sentiment_score:.2f}",
        dl_primary_intent,
        dl_secondary_intent,
        dl_emotions["grief"],
        dl_emotions["fear"],
        dl_emotions["shock"],
        dl_emotions["anger"],
        dl_emotions["sympathy"],
        dl_emotions["uncertainty"],
        dl_topics["technology"],
        dl_topics["aviation"],
        dl_topics["policy"],
    ]
}

df_cmp = pd.DataFrame(comparison)
print(df_cmp.to_string(index=False))


# ═══════════════════════════════════════════════════════════════════════════
# VISUALISATIONS
# ═══════════════════════════════════════════════════════════════════════════
COLORS = {
    "llm":  "#4C72B0",
    "dl":   "#DD8452",
    "neg":  "#C44E52",
    "neu":  "#8C8C8C",
    "pos":  "#55A868",
    "bg":   "#F8F8F8",
    "grid": "#E0E0E0"
}

fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor(COLORS["bg"])
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.38)

# ── Plot 1: VADER compound per sentence ─────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor(COLORS["bg"])
bar_colors = [COLORS["neg"] if c <= -0.05 else
              COLORS["pos"] if c >= 0.05 else COLORS["neu"]
              for c in compound_scores]
ax1.bar(range(len(compound_scores)), compound_scores, color=bar_colors,
        edgecolor="white", linewidth=0.5)
ax1.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax1.axhline(overall_compound, color=COLORS["llm"], linewidth=1.8,
            linestyle="-", alpha=0.8, label=f"Article mean={overall_compound:.3f}")
ax1.set_xlabel("Sentence index", fontsize=10)
ax1.set_ylabel("VADER compound", fontsize=10)
ax1.set_title("Sentence-level VADER Sentiment (Deep Learning baseline)", fontsize=12, fontweight="bold")
ax1.set_ylim(-1.1, 1.1)
ax1.grid(axis="y", color=COLORS["grid"], linewidth=0.5)
patches = [mpatches.Patch(color=COLORS["neg"], label="Negative"),
           mpatches.Patch(color=COLORS["neu"], label="Neutral"),
           mpatches.Patch(color=COLORS["pos"], label="Positive")]
ax1.legend(handles=patches + [plt.Line2D([0],[0], color=COLORS["llm"],
           linewidth=2, label=f"Mean={overall_compound:.3f}")],
           loc="upper right", fontsize=8, framealpha=0.7)

# ── Plot 2: Sentiment scores comparison ─────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
ax2.set_facecolor(COLORS["bg"])
methods  = ["LLM\n(Claude)", "VADER\n(DL)", "MLP\n(DL)"]
scores_2 = [llm_result["sentiment"]["score"], overall_compound, mlp_sentiment_score]
bar_c2   = [COLORS["neg"] if s < -0.05 else COLORS["pos"] if s > 0.05 else COLORS["neu"]
            for s in scores_2]
bars = ax2.bar(methods, scores_2, color=bar_c2, edgecolor="white", width=0.5)
for bar, val in zip(bars, scores_2):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{val:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax2.set_ylim(-1.1, 0.3)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax2.set_ylabel("Score (−1 = most negative)", fontsize=9)
ax2.set_title("Sentiment Score\nComparison", fontsize=11, fontweight="bold")
ax2.grid(axis="y", color=COLORS["grid"], linewidth=0.5)

# ── Plot 3: Topic breakdown ─────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
ax3.set_facecolor(COLORS["bg"])
topics_  = ["Technology", "Aviation", "Policy"]
llm_t    = [llm_result["topics"]["technology"],
             llm_result["topics"]["aviation"],
             llm_result["topics"]["policy"]]
dl_t     = [dl_topics["technology"], dl_topics["aviation"], dl_topics["policy"]]
x3       = np.arange(len(topics_))
w3       = 0.35
ax3.bar(x3 - w3/2, llm_t, w3, label="LLM",  color=COLORS["llm"], edgecolor="white")
ax3.bar(x3 + w3/2, dl_t,  w3, label="DL",   color=COLORS["dl"],  edgecolor="white")
for xi, (lv, dv) in zip(x3, zip(llm_t, dl_t)):
    ax3.text(xi - w3/2, lv + 1, str(lv), ha="center", va="bottom", fontsize=8)
    ax3.text(xi + w3/2, dv + 1, str(dv), ha="center", va="bottom", fontsize=8)
ax3.set_xticks(x3)
ax3.set_xticklabels(topics_, fontsize=9)
ax3.set_ylabel("Score (0–100)", fontsize=9)
ax3.set_title("Topic Scores\nLLM vs DL", fontsize=11, fontweight="bold")
ax3.set_ylim(0, 110)
ax3.legend(fontsize=8)
ax3.grid(axis="y", color=COLORS["grid"], linewidth=0.5)

# ── Plot 4: Emotion radar ────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2], polar=True)
emotions_list = ["Grief", "Fear", "Shock", "Anger", "Sympathy", "Uncertainty", "Hope"]
llm_emo = [llm_result["emotions"][e.lower()] for e in emotions_list]
dl_emo  = [dl_emotions[e.lower()] for e in emotions_list]
N = len(emotions_list)
angles = [n / float(N) * 2 * math.pi for n in range(N)]
angles += angles[:1]
llm_emo_c = llm_emo + llm_emo[:1]
dl_emo_c  = dl_emo  + dl_emo[:1]
ax4.set_theta_offset(math.pi / 2)
ax4.set_theta_direction(-1)
ax4.set_xticks(angles[:-1])
ax4.set_xticklabels(emotions_list, fontsize=8)
ax4.set_ylim(0, 100)
ax4.plot(angles, llm_emo_c, color=COLORS["llm"], linewidth=2, label="LLM")
ax4.fill(angles, llm_emo_c, color=COLORS["llm"], alpha=0.15)
ax4.plot(angles, dl_emo_c,  color=COLORS["dl"],  linewidth=2, linestyle="--", label="DL")
ax4.fill(angles, dl_emo_c,  color=COLORS["dl"],  alpha=0.10)
ax4.set_title("Emotion Radar\nLLM vs DL", fontsize=11, fontweight="bold", pad=15)
ax4.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8)

# ── Plot 5: MLP training loss ────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, 0])
ax5.set_facecolor(COLORS["bg"])
ax5.plot(losses, color=COLORS["dl"], linewidth=1.5)
ax5.set_xlabel("Epoch", fontsize=9)
ax5.set_ylabel("MSE Loss", fontsize=9)
ax5.set_title("PyTorch MLP\nTraining Loss", fontsize=11, fontweight="bold")
ax5.grid(color=COLORS["grid"], linewidth=0.5)

# ── Plot 6: Intent keyword hits ─────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 1])
ax6.set_facecolor(COLORS["bg"])
intents_sorted = sorted(intent_scores.items(), key=lambda x: -x[1])
intent_labels  = [i[0] for i in intents_sorted]
intent_vals    = [i[1] for i in intents_sorted]
intent_colors  = [COLORS["llm"] if i == dl_primary_intent else
                  COLORS["dl"]  if i == dl_secondary_intent else
                  "#AAAAAA" for i in intent_labels]
ax6.barh(intent_labels, intent_vals, color=intent_colors, edgecolor="white")
ax6.set_xlabel("Keyword hits", fontsize=9)
ax6.set_title("DL Intent Detection\n(keyword frequency)", fontsize=11, fontweight="bold")
ax6.grid(axis="x", color=COLORS["grid"], linewidth=0.5)
for i, v in enumerate(intent_vals):
    ax6.text(v + 0.3, i, str(v), va="center", fontsize=8)

# ── Plot 7: Emotion LLM vs DL bar chart ─────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 2])
ax7.set_facecolor(COLORS["bg"])
x7 = np.arange(len(emotions_list))
w7 = 0.38
ax7.bar(x7 - w7/2, llm_emo, w7, label="LLM", color=COLORS["llm"], edgecolor="white")
ax7.bar(x7 + w7/2, dl_emo,  w7, label="DL",  color=COLORS["dl"],  edgecolor="white")
ax7.set_xticks(x7)
ax7.set_xticklabels([e[:5] for e in emotions_list], fontsize=8)
ax7.set_ylabel("Score (0–100)", fontsize=9)
ax7.set_title("Emotion Scores\nLLM vs DL", fontsize=11, fontweight="bold")
ax7.set_ylim(0, 115)
ax7.legend(fontsize=8)
ax7.grid(axis="y", color=COLORS["grid"], linewidth=0.5)

# ── Main title ───────────────────────────────────────────────────────────────
fig.suptitle(
    "DATA 622 – Homework 9  |  NLP Analysis: AP News – Boeing / Air India Crash\n"
    "LLM (Claude claude-sonnet-4) vs Deep Learning (PyTorch MLP + VADER + Lexicon)",
    fontsize=13, fontweight="bold", y=0.98
)

out_path = "/mnt/user-data/outputs/hw9_analysis.png"
# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(out_path), exist_ok=True)
plt.savefig(out_path, dpi=160, bbox_inches="tight", facecolor=COLORS["bg"])
plt.close()
print(f"\n✓ Visualisation saved → {out_path}")


# ═══════════════════════════════════════════════════════════════════════════
# WRITTEN COMPARISON REPORT
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("WRITTEN COMPARISON REPORT")
print("=" * 70)

report = f"""
╔══════════════════════════════════════════════════════════════════════╗
║       DATA 622 – HW9 | NLP ANALYSIS  REPORT                        ║
╚══════════════════════════════════════════════════════════════════════╝

ARTICLE  : AP News – Boeing / Air India Crash (Air India Flight 171)
URL      : apnews.com/article/boeing-aviation-aircraft-air-india-crash

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. SENTIMENT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LLM  → {llm_result["sentiment"]["overall"]}  (score = {llm_result["sentiment"]["score"]:.2f})
  {textwrap.fill(llm_result["sentiment"]["rationale"], 65, subsequent_indent="  ")}

DL   → VADER: {overall_vader_label}  (mean compound = {overall_compound:.4f})
       MLP predicted score = {mlp_sentiment_score:.4f}  →  {mlp_label}
  Both tools agree the article is strongly Negative. VADER captured
  this via its lexicon of negative affect words ("killed", "crash",
  "destroyed", "fatal").  The PyTorch MLP, trained on VADER pseudo-
  labels via TF-IDF features, reproduced a similar score via learned
  feature weights, validating the self-supervised DL pipeline.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2. INTENT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LLM  → Primary: {llm_result["intent"]["primary"]}  |  Secondary: {llm_result["intent"]["secondary"]}
  {textwrap.fill(llm_result["intent"]["rationale"], 65, subsequent_indent="  ")}

DL   → Primary: {dl_primary_intent}  |  Secondary: {dl_secondary_intent}
  Keyword-frequency heuristics found the highest hit counts for
  "Inform" (verbs of reporting) and "{dl_secondary_intent}" (words like
  "concern / failure / scrutiny").  This matches the LLM judgment.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
3. EMOTIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LLM  → Grief={llm_result["emotions"]["grief"]} | Fear={llm_result["emotions"]["fear"]} | Shock={llm_result["emotions"]["shock"]} | Anger={llm_result["emotions"]["anger"]}
       Sympathy={llm_result["emotions"]["sympathy"]} | Uncertainty={llm_result["emotions"]["uncertainty"]} | Hope={llm_result["emotions"]["hope"]}
  {textwrap.fill(llm_result["emotions"]["rationale"], 65, subsequent_indent="  ")}

DL   → Grief={dl_emotions["grief"]} | Fear={dl_emotions["fear"]} | Shock={dl_emotions["shock"]} | Anger={dl_emotions["anger"]}
       Sympathy={dl_emotions["sympathy"]} | Uncertainty={dl_emotions["uncertainty"]} | Hope={dl_emotions["hope"]}
  Both approaches rank Shock and Grief highest, reflecting the mass-
  casualty language. Hope is the lowest in both – the article ends
  with safety concerns, not recovery optimism.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
4. TOPIC PROPORTIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
         Technology  Aviation   Policy
LLM  →    {llm_result["topics"]["technology"]:3d}%        {llm_result["topics"]["aviation"]:3d}%      {llm_result["topics"]["policy"]:3d}%
DL   →    {dl_topics["technology"]:3d}%        {dl_topics["aviation"]:3d}%      {dl_topics["policy"]:3d}%
  {textwrap.fill(llm_result["topics"]["rationale"], 65, subsequent_indent="  ")}

  Aviation dominates in both methods.  The DL model assigns higher
  Technology weight than the LLM because technical terms like "fuel
  control switch", "MCAS", and "flight recorder" are over-represented
  in the seed vocabulary.  The LLM contextualises these as aviation
  hardware rather than pure technology discussion.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
5. METHODOLOGY COMPARISON
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  LLM strengths  : Holistic contextual understanding; nuanced scores;
                   can disentangle overlapping topics; produces human-
                   readable rationale; no hand-crafted features needed.
  LLM limits     : Black-box reasoning; scores not reproducible across
                   calls; can hallucinate; high cost per query.

  DL  strengths  : Transparent, reproducible pipeline; fast on large
                   corpora; VADER lexicon is validated; MLP learns
                   feature weights; cosine similarity is interpretable.
  DL  limits     : Lacks world-knowledge; seed-word dependency; VADER
                   misses nuanced negation/context; MLP trained on only
                   ~{len(augmented)} samples – very limited.

  AGREEMENT      : Both approaches converge on Negative sentiment,
                   Inform+Warn intent, Shock/Grief as top emotions, and
                   Aviation as the dominant topic – suggesting robust
                   signal in the article that survives method changes.
"""
print(report)

# Save report
with open("/mnt/user-data/outputs/hw9_report.txt", "w") as f:
    f.write(report)
print("✓ Text report saved → /mnt/user-data/outputs/hw9_report.txt")

DATA 622 – Homework 9  |  Sentiment / Intent / Emotion / Topic NLP

Article: 396 words, 18 sentences

──────────────────────────────────────────────────────────────────────
APPROACH 1 – LLM (Claude claude-sonnet-4-20250514)
──────────────────────────────────────────────────────────────────────
✗ Claude API error: 'content'

──────────────────────────────────────────────────────────────────────
APPROACH 2 – Deep Learning (PyTorch MLP + VADER + TF-IDF)
──────────────────────────────────────────────────────────────────────

VADER – Overall compound score : -0.1802  →  Negative
  Negative sentences : 9/18
  Neutral  sentences : 5/18
  Positive sentences : 4/18

PyTorch MLP – Predicted sentiment score : -0.4744  →  Negative
  Training loss (final 10 epochs avg)   : 0.002925

DL Topic Scores (cosine-scaled 0-100):
  Technology  : 38
  Aviation    : 85
  Policy      : 54

DL Emotion Scores (lexicon-frequency, 0-100):
  Grief          : 100
  Fear           : 78
  Shock          : 78
  Anger  